In [1]:
!uv pip install altair

Audited 1 package in 22ms


In [2]:
import polars as pl
from polars import col


pl.Config.set_tbl_rows(1000)      # Allow rendering up to 1000 rows inline
pl.Config.set_tbl_cols(20)        # Allow rendering up to 20 columns wide
pl.Config.set_fmt_str_lengths(50) # Prevent text strings from getting cropped

market_response_path = "/Users/oceansxyz/alphanode-metal/logs/market/market_20260614_105506.jsonl"

#### Load JSONL - Binace 

In [3]:
#Raw binance reponse for market web socket - pretty much a static
book_ticker_schema = pl.Struct([
    pl.Field("ts", pl.Int64),
    pl.Field("frame", pl.Struct([
        pl.Field("u", pl.Int64),
        pl.Field("s", pl.String),
        pl.Field("b", pl.String),
        pl.Field("B", pl.String),
        pl.Field("a", pl.String),
        pl.Field("A", pl.String),
    ])),
])

with open(market_response_path) as f:
    metadata_json = f.readline().strip()
    print("Metadata Raw Line:", metadata_json)
    df_market = (
        pl.read_csv(
            f,
            has_header=False,
            new_columns=["json_str"],
            separator="\n",
            infer_schema=False,
        )
        .select(pl.col("json_str").str.json_decode(dtype=book_ticker_schema))
        .unnest("json_str")
        .unnest("frame")
        .with_columns([
            pl.col("u").cast(pl.Int64),
            pl.col("s"),
            pl.col("b").cast(pl.Float64),
            pl.col("B").cast(pl.Float64),
            pl.col("a").cast(pl.Float64),
            pl.col("A").cast(pl.Float64),
        ])
    )


# Select relevant cols for alpha calulation with identifiable file names
market_rates = df_market.select(
    col("ts"), col("s").alias("symbol"), col("b").alias("bid"), col("a").alias("ask")
)

# duration of web socket listen
min_ts = market_rates.min().select("ts").item()
max_ts = market_rates.max().select("ts").item()
duration_m =(max_ts-min_ts)/60000


print(f"Duration of listen : {duration_m} mins")
print(market_rates.group_by("symbol").len())

Metadata Raw Line: {"ts":1781434506715,"frame":{"result":null,"id":1}}
Duration of listen : 9.4146 mins
shape: (3, 2)
┌──────────┬──────┐
│ symbol   ┆ len  │
│ ---      ┆ ---  │
│ str      ┆ u32  │
╞══════════╪══════╡
│ BTCEURI  ┆ 115  │
│ EURIUSDT ┆ 8    │
│ BTCUSDT  ┆ 2966 │
└──────────┴──────┘


In [4]:
## Split dataframe symbolwise symbol is an orderbook
symbol_0 = market_rates.filter(col("symbol")=="BTCUSDT").select(["ts", col("bid").alias("BTCUSDT_bid"), col("ask").alias("BTCUSDT_ask")])
symbol_1 = market_rates.filter(col("symbol")=="BTCEURI").select(["ts", col("bid").alias("BTCEURI_bid"), col("ask").alias("BTCEURI_ask")])
symbol_2 = market_rates.filter(col("symbol")=="EURIUSDT").select(["ts", col("bid").alias("EURIUSDT_bid"), col("ask").alias("EURIUSDT_ask")])

In [5]:
# Forward fill missing values by copying the last known valid price down 
# for all generated ticker metrics simultaneously, skipping the timestamp.
market_rates_widened = symbol_0.join(symbol_1, on="ts", how="full", coalesce=True)\
    .join(symbol_2, on="ts", how="full", coalesce=True)\
    .sort("ts")\
    .select([
        col("ts"),
        col("*").exclude("ts").fill_null(strategy="forward")
    ]).drop_nulls()

market_rates_widened.head()

ts,BTCUSDT_bid,BTCUSDT_ask,BTCEURI_bid,BTCEURI_ask,EURIUSDT_bid,EURIUSDT_ask
i64,f64,f64,f64,f64,f64,f64
1781434666277,64531.44,64531.45,55825.34,55847.46,1.1556,1.1559
1781434666590,64531.44,64531.45,55825.34,55847.46,1.1556,1.1559
1781434666781,64531.44,64531.45,55825.34,55847.46,1.1556,1.1559
1781434667338,64531.44,64531.45,55825.34,55847.46,1.1556,1.1559
1781434667587,64531.44,64531.45,55825.34,55847.46,1.1556,1.1559


##### Notation & Model

NODE_0 = USDT NODE_1 = BTC NODE_2 = EURI  

EDGE_0 = BTCUSDT  EDGE_1 = BTCEURI  EDGE_2 = EURIUSDT  

Clockwise Walk  0→1  →1→2  →2→0  

BUY  BTCUSDT  @ASK  === 0→1  
SELL BTCEURI  @BID  === 1→2  
SELL EURIUSDT @BID  === 2→0


#### Clockwise Walk  0→1  →1→2  →2→0

In [13]:
clockwise_walk = market_rates_widened.select("ts", col("BTCUSDT_ask"), "BTCEURI_bid", "EURIUSDT_bid")\
    .unique(subset=["BTCUSDT_ask", "BTCEURI_bid", "EURIUSDT_bid"])\
    .with_columns((col("BTCEURI_bid")*col("EURIUSDT_bid")).alias("BTCEURIUSDT_bid"))\
    .with_columns((col("BTCEURIUSDT_bid")/col("BTCUSDT_ask")).abs().alias("alpha"))\
    .with_columns(((col("alpha")-1)/(1e-4)).alias("alpha_bps"))



clockwise_walk.sort("alpha", descending=True).head()



ts,BTCUSDT_ask,BTCEURI_bid,EURIUSDT_bid,BTCEURIUSDT_bid,alpha,alpha_bps
i64,f64,f64,f64,f64,f64,f64
1781434829285,64500.01,55823.26,1.1557,64514.941582,1.000231,2.314974
1781434826591,64500.01,55823.26,1.1556,64509.359256,1.000145,1.449497
1781434814303,64504.01,55823.4,1.1556,64509.52104,1.000085,0.854372
1781434814597,64504.02,55823.4,1.1556,64509.52104,1.000085,0.852821
1781434814598,64504.03,55823.4,1.1556,64509.52104,1.000085,0.851271


#### AntiClockwise Walk   0←1 ←1←2  ←2←0

Remarkably profitable - SELL BTCUSDT; BUY BTCEURI; and BUY EURIUSDT

In [16]:
anticlockwise_walk = market_rates_widened.select("ts", col("BTCUSDT_bid"), "BTCEURI_ask", "EURIUSDT_ask")\
    .unique(subset=["BTCUSDT_bid", "BTCEURI_ask", "EURIUSDT_ask"])\
    .with_columns((col("BTCEURI_ask")*col("EURIUSDT_ask")).alias("BTCEURIUSDT_ask"))\
    .with_columns((col("BTCEURIUSDT_ask")/col("BTCUSDT_bid")).abs().alias("alpha"))\
    .with_columns(((col("alpha")-1)/(1e-4)).alias("alpha_bps"))



anticlockwise_walk.sort("alpha", descending=True).head(20)

ts,BTCUSDT_bid,BTCEURI_ask,EURIUSDT_ask,BTCEURIUSDT_ask,alpha,alpha_bps
i64,f64,f64,f64,f64,f64,f64
1781435058304,64476.62,55900.0,1.1559,64614.81,1.002143,21.432575
1781435058599,64489.88,55900.0,1.1559,64614.81,1.001937,19.372032
1781435059590,64491.99,55900.0,1.1559,64614.81,1.001904,19.044225
1781434837301,64500.0,55900.0,1.1559,64614.81,1.00178,17.8
1781434849294,64500.0,55885.3,1.1559,64597.81827,1.001517,15.165623
1781434884302,64517.89,55900.0,1.1559,64614.81,1.001502,15.02219
1781434849598,64504.62,55885.3,1.1559,64597.81827,1.001445,14.448309
1781434850597,64507.22,55885.3,1.1559,64597.81827,1.001404,14.044671
1781434891294,64517.89,55892.14,1.1559,64605.724626,1.001361,13.613995


In [10]:
clockwise_walk.plot.line(x="ts", y="alpha")

alt.Chart(...)